# 04 - Brain MRI: EDA and Data Preparation


> **Academic prototype.** This notebook is part of a university final project.
> The models here are **not** medical devices, are **not** validated on clinical
> data, and must **never** be used to diagnose, screen or triage real patients.
> See `docs/ETHICS.md`.


**Goal:** discover image/mask pairs, **validate every pair**, understand the
foreground imbalance, and build a patient-level split.

Pair validation is the most important cell in this notebook. A mis-paired dataset
still trains without error - the loss falls a little and Dice stays near zero -
and the cause can take days to find.

**Before running:** point `data.root`, `data.layout` and `data.mask_suffix` in
`configs/mri_unet.yaml` at your dataset (see `data/README.md`).

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

print("Project root:", PROJECT_ROOT)

In [ ]:
import numpy as np
import pandas as pd

from src.common import ensure_dir, load_config, seed_everything
from src.common.io_utils import save_csv, save_json, save_figure
from src.common.viz import overlay_mask, plot_image_grid, set_plot_style
from src.segmentation import (
    assert_no_patient_leakage, build_pairs, patient_level_split_seg,
    seg_split_summary, validate_pairs,
)

set_plot_style()
pd.set_option("display.width", 140)

cfg = load_config("mri_unet.yaml")
seed_everything(cfg.get("seed", 42), deterministic=cfg.get("deterministic", True))

FIG_DIR = ensure_dir(PROJECT_ROOT / "outputs/segmentation/eda/figures")
MET_DIR = ensure_dir(PROJECT_ROOT / "outputs/segmentation/eda/metrics")

print("Dataset root :", cfg.path("data.root"))
print("Layout       :", cfg.get("data.layout"), "| mask suffix:", cfg.get("data.mask_suffix"))
print("Image size   :", cfg.get("data.image_size"))

## 1. Discover image/mask pairs

If this raises, the error message shows what file names were actually found and which config key to change.

In [ ]:
pairs = build_pairs(cfg)
print(f"{len(pairs)} pairs across {pairs['patient_id'].nunique()} patients\n")
pairs.head()

## 2. Validate every pair

Checks performed on each pair: both files readable, identical height/width, mask
effectively binary. It also records the tumour pixel fraction per slice, which
drives the imbalance analysis below.

**Screenshot this output for the report** - it is the evidence that the pairing
is correct.

In [ ]:
validation, summary = validate_pairs(pairs, sample=None)
save_csv(validation, MET_DIR / "pair_validation.csv")
save_json(summary, MET_DIR / "pair_validation_summary.json")

In [ ]:
# Keep only valid pairs, and carry the per-slice statistics forward for stratification.
pairs = validation[validation["ok"]].reset_index(drop=True)
print(f"{len(pairs)} valid pairs kept ({summary['n_problems']} dropped)")

if summary["n_problems"]:
    print("\nDropped pairs (first 5):")
    for item in summary["problems_sample"][:5]:
        print(f"  {Path(item['image_path']).name}: {item['problem']}")

## 3. Image dimensions

In [ ]:
sizes = pairs.groupby(["image_h", "image_w"]).size().sort_values(ascending=False)
print("Slice dimensions (height x width):")
print(sizes.to_string())
print("\nChannels:", pairs["image_channels"].value_counts().to_dict())
print("Image dtypes:", pairs["image_dtype"].value_counts().to_dict())
print(f"\nAll slices are resized to {cfg.get('data.image_size')}x{cfg.get('data.image_size')} at load time.")

## 4. Class imbalance: how much of a slice is tumour?

This is the central difficulty of the task. Tumours usually cover a very small
fraction of a slice, and many slices contain no tumour at all. That is why the
loss is Dice + BCE rather than plain BCE, and why pixel accuracy is not reported
at all - "all background" would score above 98%.

In [ ]:
import matplotlib.pyplot as plt

tumour_slices = pairs[pairs["has_tumour"] == 1]
fraction = tumour_slices["tumour_pixel_fraction"]

print(f"Slices total          : {len(pairs)}")
print(f"Slices with tumour    : {len(tumour_slices)} ({len(tumour_slices) / len(pairs):.1%})")
print(f"Slices without tumour : {len(pairs) - len(tumour_slices)}")
print(f"\nTumour pixel fraction on tumour-containing slices:")
print(fraction.describe().round(5).to_string())
print(f"\nA constant 'all background' prediction would reach "
      f"{1 - pairs['tumour_pixel_fraction'].mean():.2%} pixel accuracy "
      f"and a Dice of 0 on tumour slices.")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(["with tumour", "no tumour"],
            [len(tumour_slices), len(pairs) - len(tumour_slices)],
            color=["#C44E52", "#4C72B0"], edgecolor="black", linewidth=0.5)
axes[0].set_ylabel("Number of slices")
axes[0].set_title("Slices with vs without tumour")
axes[1].hist(fraction * 100, bins=40, color="#C44E52", edgecolor="black", linewidth=0.5)
axes[1].set_xlabel("Tumour area (% of slice)")
axes[1].set_ylabel("Number of slices")
axes[1].set_title("Tumour coverage (tumour-containing slices)")
fig.tight_layout()
save_figure(fig, FIG_DIR / "mask_coverage.png", close=False)

## 5. Tumour burden per patient

Some patients contribute many tumour slices, others few. This is why the split is stratified on whether a patient has any tumour at all.

In [ ]:
per_patient = pairs.groupby("patient_id").agg(
    n_slices=("image_path", "size"),
    n_tumour_slices=("has_tumour", "sum"),
    mean_coverage=("tumour_pixel_fraction", "mean"),
).sort_values("n_tumour_slices", ascending=False)

print(per_patient.describe().round(3).to_string())
print(f"\nPatients with no tumour slice at all: {(per_patient['n_tumour_slices'] == 0).sum()}")
per_patient.head(10).round(5)

## 6. Sample MRI slices with mask overlays

Green = annotated tumour. Look for masks that are obviously offset from the visible lesion - that would indicate a pairing or orientation problem that the automated checks cannot catch.

In [ ]:
from PIL import Image

rng = np.random.default_rng(cfg.get("seed", 42))
picks = tumour_slices.iloc[rng.choice(len(tumour_slices), size=min(6, len(tumour_slices)),
                                      replace=False)]

overlays, titles = [], []
for record in picks.itertuples(index=False):
    image = np.array(Image.open(record.image_path).convert("RGB"))
    mask = np.array(Image.open(record.mask_path).convert("L"))
    overlays.append(overlay_mask(image, mask > 127, color=(0, 1, 0), alpha=0.45))
    titles.append(f"{record.patient_id}\n{record.tumour_pixel_fraction:.2%} tumour")

fig = plot_image_grid(overlays, titles, ncols=3,
                      suptitle="MRI slices with ground-truth tumour masks (green)")
save_figure(fig, FIG_DIR / "sample_overlays.png", close=False)

In [ ]:
# Side-by-side view of one pair: raw slice next to its mask.
record = picks.iloc[0]
image = np.array(Image.open(record["image_path"]).convert("RGB"))
mask = np.array(Image.open(record["mask_path"]).convert("L"))

fig = plot_image_grid(
    [image, mask > 127, overlay_mask(image, mask > 127, color=(0, 1, 0), alpha=0.45)],
    ["MRI slice", "Ground-truth mask", "Overlay"],
    ncols=3, suptitle=f"Pair check: {Path(record['image_path']).name}",
)
save_figure(fig, FIG_DIR / "pair_example.png", close=False)

## 7. Patient-level split

Adjacent slices from the same MRI volume are nearly identical images. A random
slice-level split would place near-duplicates in both train and test and inflate
Dice substantially. Splitting is therefore done **by patient**, stratified on
whether the patient has any tumour slices.

In [ ]:
split_df = patient_level_split_seg(
    pairs,
    val_size=float(cfg.get("data.val_size", 0.15)),
    test_size=float(cfg.get("data.test_size", 0.15)),
    seed=int(cfg.get("seed", 42)),
    stratify_on_tumour=bool(cfg.get("data.stratify_on_tumour", True)),
)
leakage = assert_no_patient_leakage(split_df)
save_json(leakage, MET_DIR / "split_leakage_check.json")

In [ ]:
summary_table = seg_split_summary(split_df)
save_csv(summary_table, MET_DIR / "split_summary.csv")
summary_table

## 8. Save the prepared pair table

Notebooks 05 and 06 read this file, so the split is fixed once and reused.

In [ ]:
PAIRS_PATH = ensure_dir(PROJECT_ROOT / "data/processed") / "mri_pairs.csv"
save_csv(split_df, PAIRS_PATH)

print("\nColumns:", list(split_df.columns))
print(f"\nNext: run 05_mri_unet_training.ipynb (it loads {PAIRS_PATH.name})")

---

## Findings to write up

- number of slices, patients, and slices per patient;
- percentage of slices containing tumour;
- mean tumour area as a percentage of the slice - the imbalance figure that
  justifies Dice + BCE;
- any pairs dropped and why;
- split strategy and leakage-check result.

### Screenshots for the report
- Section 2 pair-validation output
- Section 4 mask-coverage figures
- Section 6 sample overlays
- Section 7 leakage check and split summary